In [9]:
import opt_einsum as oe
import numpy as np
import torch
import sys
sys.path.append("../../")
import mps
from mps.trainer.data_utils import SyntheticDataset
from mps.torchmps.torchmps import MPS

In [46]:
N = 1000
dataset = SyntheticDataset(n=N, num_samples=128, seed=42)

# Balance the dataset to have equal number of samples for each label
labels = [sample[1] for sample in dataset]
label_0_indices = [i for i, label in enumerate(labels) if label == 0]
label_1_indices = [i for i, label in enumerate(labels) if label == 1]

# Determine the number of samples to balance the dataset
num_samples_per_label = min(len(label_0_indices), len(label_1_indices))

# Create balanced indices
balanced_indices = label_0_indices[:2] + label_1_indices[:2]

# Create a balanced dataset
balanced_dataset = torch.utils.data.Subset(dataset, balanced_indices)
dataloader = torch.utils.data.DataLoader(balanced_dataset, batch_size=128, shuffle=True)


In [63]:
logsoftmax = torch.nn.LogSoftmax(dim=-1)
nnloss = torch.nn.NLLLoss(reduction="mean")

my_mps = MPS(input_dim=N, output_dim=2, bond_dim=100, init_std=1e-3)
optimizer = torch.optim.SGD(my_mps.parameters(), lr=0.00001)
my_mps.train()

data, target = next(iter(dataloader))

outputs = my_mps(data[:,:,0].float())


loss_list = []
for i in range(1000):
    data, target = next(iter(dataloader))
    outputs = my_mps(data[:,:,0].float())
    outputs = outputs / outputs.sum(dim=-1, keepdim=True)
    log_probs = logsoftmax(outputs)
    loss = nnloss(log_probs, target)
    loss.backward()
    optimizer.step()
    preds = log_probs.argmax(dim=-1)
    acc = (preds == target).float().mean().item()
    loss_list.append(loss.item())
    print("Accuracy: ", acc, "Loss: ", loss.item())
print(loss_list[-10:])

Accuracy:  0.5 Loss:  0.6931486129760742
Accuracy:  0.5 Loss:  0.6931486129760742
Accuracy:  0.5 Loss:  0.6931486129760742
Accuracy:  0.5 Loss:  0.6931486129760742
Accuracy:  0.5 Loss:  0.6931486129760742
Accuracy:  0.5 Loss:  0.693148672580719
Accuracy:  0.5 Loss:  0.6931486129760742
Accuracy:  0.5 Loss:  0.6931485533714294
Accuracy:  0.5 Loss:  0.6931483745574951
Accuracy:  0.5 Loss:  0.6931484937667847
Accuracy:  0.5 Loss:  0.6931484937667847
Accuracy:  0.5 Loss:  0.6931483745574951
Accuracy:  0.5 Loss:  0.6931483745574951
Accuracy:  0.5 Loss:  0.6931483745574951
Accuracy:  0.5 Loss:  0.6931483745574951
Accuracy:  0.5 Loss:  0.6931482553482056
Accuracy:  0.5 Loss:  0.6931482553482056
Accuracy:  0.5 Loss:  0.693148136138916
Accuracy:  0.5 Loss:  0.6931480169296265
Accuracy:  0.5 Loss:  0.6931478977203369
Accuracy:  0.5 Loss:  0.6931478977203369
Accuracy:  0.5 Loss:  0.6931478381156921
Accuracy:  0.5 Loss:  0.6931476593017578
Accuracy:  0.5 Loss:  0.6931476593017578
Accuracy:  0.5 Los

KeyboardInterrupt: 

In [61]:
my_mps(data[:,:, 0].float())

tensor([[1.0070],
        [1.0055],
        [1.0070],
        [1.0055]], grad_fn=<ViewBackward0>)